# 智能文本生成系统性能测试 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

## 本机 Jupyter 交付版（29 行）

这是按题目要求使用 5 个 Python 线程的精简参考实现。浏览器内核不能运行 `ThreadPoolExecutor`；请在本机 Jupyter 中运行这一版。

```python
import random, time
from concurrent.futures import ThreadPoolExecutor
class TextGeneratorSimulator:
    def __init__(self, delay=0.2, error=0.02):
        self.delay, self.error = delay, error
    def generate(self, prompt):
        time.sleep(self.delay * random.uniform(0.8, 1.2))
        if random.random() < self.error:
            raise RuntimeError("请求失败")
        return prompt + "，模拟续写"
sim = TextGeneratorSimulator()
prompts = ["天气", "电表", "巡检", "故障", "客服", "数据", "模型", "安全", "节能", "应急"]
def test(prompt):
    start = time.perf_counter()
    try:
        output, status = sim.generate(prompt), "成功"
    except RuntimeError as error:
        output, status = str(error), "失败"
    return prompt, status, (time.perf_counter()-start)*1000, output
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(test, prompts))
seconds = time.perf_counter()-start
for prompt, status, ms, output in results:
    print(prompt, status, f"{ms:.1f}ms", output)
times = [row[2] for row in results]
ok = sum(row[1] == "成功" for row in results)
print("成功率", f"{ok/len(results):.0%}", "平均耗时", f"{sum(times)/len(times):.1f}ms")
print("吞吐量", f"{len(results)/seconds:.2f}次/秒", "最短/最长", f"{min(times):.1f}/{max(times):.1f}ms")
```

## 浏览器可运行改写版

下方代码改用 `asyncio` 模拟 5 人并发，仅用于公开网页练习；正式交付时以题目要求和现场 Python 环境为准。

In [ ]:
import asyncio
import random
import time

class TextGeneratorSimulator:
    def __init__(self, delay_ms=200, error_rate=0.02):
        self.delay_ms = delay_ms
        self.error_rate = error_rate

    async def generate(self, prompt):
        await asyncio.sleep(self.delay_ms / 1000)
        if random.random() < self.error_rate:
            raise RuntimeError("模拟请求失败")
        return f"{prompt}，这是模拟续写。"

simulator = TextGeneratorSimulator()
prompts = ["今天天气", "智能电表", "设备巡检", "故障预警", "客户服务",
           "数据质量", "模型训练", "安全生产", "节能方案", "应急处置"]

async def send_request(number, prompt):
    started = time.perf_counter()
    try:
        answer = await simulator.generate(prompt)
        return {"number": number, "ok": True,
                "seconds": time.perf_counter() - started, "answer": answer}
    except Exception as error:
        return {"number": number, "ok": False,
                "seconds": time.perf_counter() - started, "error": str(error)}

def report(label, results, elapsed):
    successful = [item for item in results if item["ok"]]
    times = [item["seconds"] for item in results]
    print(label, "成功率", round(len(successful) / len(results), 2),
          "平均响应时间", round(sum(times) / len(times), 3),
          "吞吐量", round(len(results) / elapsed, 2),
          "最短", round(min(times), 3), "最长", round(max(times), 3))

async def main():
    started = time.perf_counter()
    sequential = []
    for number, prompt in enumerate(prompts, 1):
        sequential.append(await send_request(number, prompt))
    report("顺序", sequential, time.perf_counter() - started)

    semaphore = asyncio.Semaphore(5)
    async def limited(number, prompt):
        async with semaphore:
            return await send_request(number, prompt)

    started = time.perf_counter()
    concurrent = await asyncio.gather(
        *(limited(number, prompt) for number, prompt in enumerate(prompts, 1))
    )
    report("5 人并发", concurrent, time.perf_counter() - started)

await main()
